In [7]:
import torch
import torch.nn.functional as F
from torchinfo import summary

from improved_diffusion.models.vae import AbstractVAE


In [8]:
def vae_loss(x_hat, x, mu, logvar, beta=1.0, recon="mse"):
    if recon == "bce":
        rec = F.binary_cross_entropy_with_logits(x_hat, x, reduction="mean")
    else:
        rec = F.mse_loss(x_hat, x, reduction="mean")
    # KL(q(z|x) || N(0,I))
    kl = -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())
    return rec + beta * kl, rec, kl

In [9]:
vae = AbstractVAE(in_channels=1,channel_mult=(1, 2, 4, 4, 6, 6, 8), latent_dim=128, dims=2).to("cpu")
x = torch.randn(2, 1, 256,256, device="cpu")
x_hat, mu, logvar, z = vae(x)
loss, rec, kl = vae_loss(x_hat, x, mu, logvar, beta=0.1)

print(x_hat.shape,mu.shape,logvar.shape,z.shape)


summary(vae, depth = 3, input_data=(x))

3 8
4 16
5 32
torch.Size([2, 1, 256, 256]) torch.Size([2, 128]) torch.Size([2, 128]) torch.Size([2, 128])


Layer (type:depth-idx)                   Output Shape              Param #
AbstractVAE                              [2, 1, 256, 256]          --
├─Sequential: 1-1                        [2, 256, 4, 4]            --
│    └─Conv2d: 2-1                       [2, 32, 256, 256]         320
│    └─ResBlock: 2-2                     [2, 32, 256, 256]         --
│    │    └─Sequential: 3-1              [2, 32, 256, 256]         18,624
│    └─Downsample: 2-3                   [2, 64, 128, 128]         --
│    │    └─Conv2d: 3-2                  [2, 64, 128, 128]         32,832
│    └─ResBlock: 2-4                     [2, 64, 128, 128]         --
│    │    └─Sequential: 3-3              [2, 64, 128, 128]         74,112
│    └─Downsample: 2-5                   [2, 128, 64, 64]          --
│    │    └─Conv2d: 3-4                  [2, 128, 64, 64]          131,200
│    └─ResBlock: 2-6                     [2, 128, 64, 64]          --
│    │    └─Sequential: 3-5              [2, 128, 64, 64]          